# 🎯 Day 34 — Mock Interview Practice — **FAANG / Top Product Company Level**

**Week 5 · Day 5 · Tue Nov 24, 2026** · Difficulty tags: **9 × [FAANG], 9 × [HARD]** · 18 questions across 8 categories

> Natural step-up from Day 33 (differential privacy, MATCH_RECOGNIZE, HNSW, t-digest, BM25, uplift/Qini, compaction math, bucketing-vs-salting, multi-sink replay, late side-lane, sunsetting runbook, crypto-shredding, rollup lattices, Cortex+PHI, DMFs, logging platform, outage catch-up, composite health score). Day 34 pushes into **underused surfaces and sequencing semantics**: availability spines and repeatable sampling in pure SQL, MSTs and weighted reservoirs in Python, Spark-4 UDTFs and time-weighted aggregates in PySpark, Python-worker memory and RocksDB state stores under Spark, per-key FIFO DLQ replay, poison-batch bisection, Delta row-level lineage, non-event coverage facts, heterogeneous event unions, Snowflake ML functions and Marketplace economics, and a monitoring trio of cardinality, MTTR and usage-ROI.

| # | Category | Topic | Tag |
| --- | --- | --- | --- |
| Q1 | SQL Advanced | Next-available-slot scheduling (spine + anti-join) | [FAANG] |
| Q2 | SQL Advanced | Repeatable sampling: TABLESAMPLE, seeds, stratification | [HARD] |
| Q3 | Python | Kruskal MST + union-find | [FAANG] |
| Q4 | Python | Weighted reservoir sampling (Efraimidis–Spirakis A-Res) | [HARD] |
| Q5 | PySpark Coding | Spark 4 Python UDTF (user-defined table function) | [FAANG] |
| Q6 | PySpark Coding | Time-weighted averages (TWAP) over state intervals | [HARD] |
| Q7 | Spark Optimization | Python worker memory model & Arrow | [HARD] |
| Q8 | Spark Deep | State store internals: HDFS-backed vs RocksDB | [FAANG] |
| Q9 | ETL / Fault Tolerance | DLQ replay with per-key FIFO ordering | [HARD] |
| Q10 | ETL / Fault Tolerance | Poison-batch bisection | [FAANG] |
| Q11 | ETL / Fault Tolerance | Delta row-level lineage (row tracking) | [FAANG] |
| Q12 | Data Modeling | Non-events: expected-vs-actual coverage facts | [HARD] |
| Q13 | Data Modeling | Heterogeneous multi-channel event union | [FAANG] |
| Q14 | Snowflake | Snowflake ML functions (FORECAST / ANOMALY_DETECTION) | [FAANG] |
| Q15 | Snowflake | Marketplace listings & data monetization | [HARD] |
| Q16 | Monitoring | Metrics cardinality explosion | [HARD] |
| Q17 | Monitoring | MTTR analytics on system.lakeflow.job_run_timeline | [HARD] |
| Q18 | Monitoring | Usage-weighted data product ROI scorecard | [FAANG] |

**Rules of the day:** attempt each question on paper first; every runnable solution cell ends with a `Q# OK` self-check; talk-track cells end with a whiteboard-style answer you should be able to reproduce out loud.

In [0]:
# Day 34 setup — imports + per-run scratch directory (all Delta demos write under BASE)
import math, random, uuid, itertools, datetime as dt
from collections import Counter, defaultdict, deque
import pyspark.sql.functions as F
from pyspark.sql import Window

BASE = f"/Volumes/workspace/default/etl_test_volume/d34_{uuid.uuid4().hex[:8]}"
print("scratch dir:", BASE)
print("spark:", spark.version, "| serverless Spark Connect session")

## Q1 — SQL Advanced — Next-Available-Slot Scheduling (Spine + Anti-Join) `[FAANG]`

**Scenario (Google Maps booking / Practo):** a clinic wants the **next 2 open 30-minute slots per doctor after 09:30**, plus each doctor's availability ratio, from a `booked` table. There is no `slots` table — you must synthesize one.

**Requirements**
1. Generate the candidate **slot spine from the calendar** (`sequence(...)`), NOT from the bookings table — deriving the spine from booked times silently deletes slots that happen to be free.
2. `LEFT ANTI JOIN` against bookings; then a per-doctor `row_number()` window returns the next N slots strictly **after** the preferred time ('after' is `>`, not `>=`).
3. Report an **availability ratio** per doctor (open slots / spine size) and explain the trade-offs: spine density vs join size, why clustering on `slot_start` enables file pruning, and how you'd extend to finding contiguous multi-slot blocks (LAG over rn).

**Interview traps:** 'strictly after' semantics; `distinct` spine; `slot_end = slot_start + INTERVAL`; the classic wrong answer builds `doctor × booked_times` and misses every never-booked time.

In [0]:
# Solution Q1 — slot spine from the calendar, anti-join, next-N per doctor
slot_spine = spark.sql("""
  SELECT explode(sequence(to_timestamp('2026-11-24 09:00'), to_timestamp('2026-11-24 12:00'), INTERVAL 30 MINUTES)) AS slot_start
""")
doctors = spark.createDataFrame([("d1","GP"), ("d2","cardio")], "doc string, dept string")
booked = spark.createDataFrame([
    ("d1", dt.datetime(2026,11,24, 9, 0)), ("d1", dt.datetime(2026,11,24,10, 0)),
    ("d2", dt.datetime(2026,11,24,10, 0)), ("d2", dt.datetime(2026,11,24,10,30)),
    ("d2", dt.datetime(2026,11,24,11, 0)),
], "doc string, slot_start timestamp")

# 1) spine from the CALENDAR (7 slots 09:00..12:00), crossed with doctors
cands = doctors.crossJoin(slot_spine)
# 2) LEFT ANTI JOIN = free slots
open_slots = (cands.join(booked, ["doc", "slot_start"], "leftanti")
                .withColumn("slot_end", F.col("slot_start") + F.expr("INTERVAL 30 MINUTES")))
open_slots.orderBy("doc", "slot_start").show(truncate=False)

# 3) next 2 open slots per doctor strictly after 09:30
w = Window.partitionBy("doc").orderBy("slot_start")
next_slots = (open_slots.where(F.col("slot_start") > F.lit(dt.datetime(2026,11,24,9,30)))
              .withColumn("rn", F.row_number().over(w)).where("rn <= 2").drop("rn"))
next_slots.orderBy("doc", "slot_start").show(truncate=False)
r = {(row.doc, str(row.slot_start)) for row in next_slots.collect()}
assert ("d1", "2026-11-24 10:30:00") in r and ("d1", "2026-11-24 11:00:00") in r
assert ("d2", "2026-11-24 11:30:00") in r and ("d2", "2026-11-24 12:00:00") in r
assert len(r) == 4

# 4) availability ratio per doctor (spine = 7 slots)
(open_slots.groupBy("doc").agg(F.count("*").alias("open_slots"),
   (F.count("*") / F.lit(7)).alias("open_ratio")).orderBy("doc").show())
print("Q1 OK: calendar-spine + anti-join returns the true next slots (ratios 5/7, 4/7); "
      "building the spine from bookings would have hidden the free times")

## Q2 — SQL Advanced — Repeatable Sampling: TABLESAMPLE, Seeds, Stratification `[HARD]`

**Scenario (Meta / Netflix experimentation platform):** QA wants a **10% sample of a 2,000-row event table that is identical on every rerun** (so a bug repro is stable), plus a **stratified** sample that respects segment proportions.

**Requirements**
1. Compare sampling mechanisms and say which ones exist in Spark SQL: `TABLESAMPLE (p PERCENT)`, `TABLESAMPLE (n ROWS)`, `TABLESAMPLE BERNOULLI`, and `TABLESAMPLE SYSTEM` — know which raises an error on a managed table and *why* (SYSTEM samples storage blocks and needs a direct file scan).
2. Show `rand(seed)` reproducibility across two scans and explain **exactly what the seed guarantees** (stable per partition-index/row-ordering — a repartition or file layout change breaks it) and how to make it bullet-proof (`md5(concat(key, seed))` hashing).
3. Stratified sampling with `stat.sampleBy(col, fractions, seed)`; verify seed-stability and the CI math: 95% half-width = 1.96·√(p(1−p)/n) → how many rows for ±3 pct-points?

**Interview traps:** SYSTEM vs BERNOULLI row-vs-block semantics; `rand()` without seed is nondeterministic; sampling *after* a WHERE can silently change the sampled population.

In [0]:
# Solution Q2 — TABLESAMPLE zoo, rand(seed) reproducibility, stratified sampleBy, CI math
ev = spark.createDataFrame([(i, f"cust_{i%7}", f"seg{i%3}", i%5) for i in range(2000)],
      "id int, cust string, seg string, score int")
ev.createOrReplaceTempView("ev")

# 1) what Spark SQL actually supports
for stmt in ["SELECT count(*) FROM ev TABLESAMPLE (10 PERCENT)",
             "SELECT count(*) FROM ev TABLESAMPLE (20 ROWS)",
             "SELECT count(*) FROM ev TABLESAMPLE BERNOULLI (10 PERCENT)",
             "SELECT count(*) FROM ev TABLESAMPLE SYSTEM (10 PERCENT)"]:
    try:
        n = spark.sql(stmt).collect()[0][0]
        print("OK :", stmt, "->", n)
    except Exception as e:
        print("NO :", stmt, "->", type(e).__name__, str(e)[:110])

# 2) rand(seed): deterministic across scans of the same layout
def one_scan():
    return set(spark.sql("SELECT id FROM ev WHERE rand(42) < 0.1").collect())
a, b = one_scan(), one_scan()
print("rand(42) reproducible across two scans:", a == b, f"({len(a)} ids)")
assert a == b

# 3) stratified sampling — seed-stable, respects segment fractions
fracs = {"seg0": 0.5, "seg1": 0.5, "seg2": 0.5}
c1 = {r["seg"]: r["count"] for r in ev.stat.sampleBy("seg", fracs, seed=42).groupBy("seg").count().collect()}
c2 = {r["seg"]: r["count"] for r in ev.stat.sampleBy("seg", fracs, seed=42).groupBy("seg").count().collect()}
assert c1 == c2
print("sampleBy seed-stable:", c1)

# 4) sample-size math: p=0.5 worst case
def half_width(n, p=0.5): return 1.96 * math.sqrt(p*(1-p)/n)
print(f"95% half-width at n=2000: ±{half_width(2000)*100:.2f} pct-pts")
need = (1.96**2 * 0.25) / 0.03**2
print(f"rows needed for ±3 pct-points: {need:.0f} (use ceil)")
print("Q2 OK: BERNOULLI/PERCENT/ROWS work, SYSTEM is unsupported on managed tables; "
      "seeds make samples reproducible; stratification preserves proportions")

## Q3 — Python — Kruskal MST + Union-Find `[FAANG]`

**Scenario (Google fleet / Uber infra):** connect 5 datacenters with the minimum total cable cost. Classic Kruskal: sort edges by weight, greedily accept edges whose endpoints are in different components (tracked by a disjoint-set union / DSU), stop after V−1 edges.

**Requirements**
1. Implement DSU with **path halving** and union by root — near-O(1) amortized find (inverse Ackermann with rank); explain why path compression + union by rank gives that guarantee.
2. Run Kruskal on the 8-edge graph; **cross-check the total weight against brute force over all spanning trees** (choose 4 edges of 8, test connectivity).
3. Explain the **cut property** (the lightest edge across any cut belongs to some MST — the correctness argument), the O(E log E) complexity, and when **Prim** beats Kruskal (dense graphs; Prim = O(E + V log V) with a binary heap).

**Interview traps:** forgetting the early stop after V−1 edges; union-by-rank vs path compression confusion; DSU with path halving only is still correct.

In [0]:
# Solution Q3 — Kruskal MST with DSU + brute-force cross-check
edges = [(0,1,4), (0,2,1), (0,3,3), (1,2,2), (1,4,5), (2,3,7), (2,4,6), (3,4,8)]

parent = list(range(5))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]   # path halving
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb: return False
    parent[rb] = ra
    return True

mst, total = [], 0
for u, v, w in sorted(edges, key=lambda e: e[2]):
    if union(u, v):                    # accepts only cross-component edges (cut property)
        mst.append((u, v, w)); total += w
    if len(mst) == 4: break            # V-1 edges => spanning tree
print("Kruskal MST edges:", mst, "total:", total)

# brute force: every 4-edge subset that connects all 5 nodes
best = None
for combo in itertools.combinations(edges, 4):
    p = list(range(5))
    def f2(x):
        while p[x] != x: x = p[x]
        return x
    ok, tw = True, 0
    for u, v, w in combo:
        ru, rv = f2(u), f2(v)
        if ru == rv: ok = False; break
        p[rv] = ru; tw += w
    if ok and (best is None or tw < best[0]): best = (tw, combo)
print("brute-force MST weight:", best[0], sorted(best[1]))
assert best[0] == total == sum(w for _, _, w in mst)
assert len(mst) == 4
print("Q3 OK: Kruskal == brute-force optimum; DSU find/union with path halving; "
      "complexity O(E log E) dominated by the edge sort")

## Q4 — Python — Weighted Reservoir Sampling (Efraimidis–Spirakis A-Res) `[HARD]`

**Scenario (Meta ads / Netflix artwork):** pick k items **without replacement** with probability proportional to campaign weights — and do it in a **single pass** over a stream you cannot hold in memory.

**Requirements**
1. Implement **A-Res**: each item gets key `u^(1/w)` with u ~ U(0,1); keep the k **largest** keys. Prove empirically that the k=1 inclusion probability equals the normalized weight, and cross-check k=2 inclusion probabilities against the **exact Plackett–Luce** probabilities computed by enumeration.
2. Add the **streaming variant** with a size-k min-heap (a new item only enters if its key beats the smallest key in the heap) — O(n log k) total.
3. Explain why the **largest**-key direction is right: `-ln(u)/w ~ Exp(w)`, so the largest `u^(1/w)` is the smallest exponential clock — an exponential race whose winner probability is `w/W` (Plackett–Luce). State the classic failure: keeping the smallest keys gives an **anti-weighted** sample.

**Interview traps:** taking the k smallest keys (inverts weights); reservoir sampling assumes arrival order independent of weights; w=0 gives an undefined exponent.

In [0]:
# Solution Q4 — A-Res: keep k LARGEST keys u^(1/w)  (== smallest Exp(w) clocks == Plackett-Luce)
items = [("ad_A", 0.10), ("ad_B", 0.30), ("ad_C", 0.05), ("ad_D", 0.45), ("ad_E", 0.10)]

def a_res_sample(items, k, rng):
    keyed = sorted(((rng.random() ** (1.0 / w), it) for it, w in items), key=lambda t: -t[0])
    return [it for _, it in keyed[:k]]          # LARGEST keys, not smallest!

# 1) k=1: inclusion frequency == normalized weight
random.seed(9)
N = 20000
cnt = Counter(a_res_sample(items, 1, random)[0] for _ in range(N))
emp = {it: cnt[it]/N for it, _ in items}
print("k=1 empirical vs target:", {it: (round(emp[it],3), w) for it, w in items})
for it, w in items:
    assert abs(emp[it] - w) < 0.02, (it, emp[it], w)

# 2) k=2 on a 4-item set: exact Plackett-Luce inclusion via DP over ordered picks
def pl_incl(w, k):
    inc = [0.0]*len(w); stack = [([], 1.0)]
    while stack:
        seq, p = stack.pop()
        if len(seq) == k:
            for i in seq: inc[i] += p
            continue
        rem = [i for i in range(len(w)) if i not in seq]
        tot = sum(w[i] for i in rem)
        for i in rem: stack.append((seq+[i], p * w[i]/tot))
    return inc
small = [("a",0.1), ("b",0.2), ("c",0.3), ("d",0.4)]
inc = Counter(); M = 30000
for _ in range(M):
    for it in a_res_sample(small, 2, random): inc[it] += 1
emp2 = {it: inc[it]/M for it, _ in small}
exact = pl_incl([0.1,0.2,0.3,0.4], 2)
print("k=2 empirical:", {it: round(emp2[it],3) for it, _ in small},
      "| exact PL:", {it: round(e,3) for (it,_), e in zip(small, exact)})
for (nm, _), e in zip(small, exact):
    assert abs(emp2[nm] - e) < 0.02, (nm, emp2[nm], e)

# 3) streaming single-pass variant: min-heap of size k
import heapq
def a_res_stream(stream, k):
    heap = []
    for it, w in stream:
        key = random.random() ** (1.0 / w)
        if len(heap) < k: heapq.heappush(heap, (key, it))
        elif key > heap[0][0]: heapq.heapreplace(heap, (key, it))
    return sorted(it for _, it in heap)
out = a_res_stream([("x",1.0),("y",5.0),("z",1.0),("w",5.0),("v",1.0),("u",1.0)], 2)
assert len(out) == 2 and len(set(out)) == 2
print("streaming A-Res k=2 sample:", out, "(heap holds only k keys — O(n log k))")
print("Q4 OK: A-Res inclusion == weights at k=1 and == exact Plackett-Luce at k=2; "
      "largest-key direction validated empirically")

## Q5 — PySpark Coding — Spark 4 Python UDTF (User-Defined Table Function) `[FAANG]`

**Scenario (Databricks / Adobe telemetry):** a monitoring rule engine needs each device row to explode into **zero or more diagnostic rows** (temperature breach, low RPM, an audit OK row) — a row-to-table mapping, not a row-to-row projection. Spark 4.0 shipped **Python UDTFs** for exactly this.

**Requirements**
1. Define a UDTF with the `@udtf(returnType=...)` decorator: an optional `__init__`, an `eval()` that **yields** tuples (called once per input row), and a static `terminate()` that yields partition-level summary rows.
2. Invoke it three ways: standalone with `lit()` constants; per-row with columns via the **`DataFrame.lateralJoin()`** API; and explain why `LATERAL TABLE(fn(...))` in SQL raises UNRESOLVED_ROUTINE for a Python-decorated UDTF (it is not registered in the catalog — register it as a SQL function first).
3. Say when a UDTF beats `explode()` + `when/otherwise` (stateful partition logic, polymorphic outputs, external calls in `terminate`) and the cost (Python serialization per row — measure before adopting at scale).

**Interview traps:** `eval` must be a generator (use `yield`); `terminate` must be `@staticmethod`; column references in SQL need catalog registration.

In [0]:
# Solution Q5 — Spark 4 Python UDTF: device diagnostics explosion
from pyspark.sql.functions import udtf, lit

@udtf(returnType="metric string, sev string, detail string")
class HealthExplode:
    def __init__(self):
        self.n = 0                       # partition-level state
    def eval(self, device: str, temp_c: float, rpm: int):
        self.n += 1
        if temp_c > 80: yield ("temp", "HIGH", f"{device} at {temp_c}C")
        if rpm < 1000: yield ("rpm", "LOW", f"{device} rpm {rpm}")
        yield ("checked", "OK", f"{device} sample {self.n}")
    @staticmethod
    def terminate():
        yield ("batch", "SUMMARY", "end of partition")

machines = spark.createDataFrame([("m1", 85.2, 950), ("m2", 40.0, 1500), ("m3", 91.0, 800)],
                                 "device string, temp_c double, rpm int")
machines.createOrReplaceTempView("machines")

# 1) standalone invocation with literals — returns its own DataFrame
out = HealthExplode(lit("m1"), lit(85.2), lit(950))
out.show(truncate=False)

# 2) per-row invocation: DataFrame.lateralJoin keeps the input columns
joined = machines.lateralJoin(HealthExplode(F.col("device"), F.col("temp_c"), F.col("rpm")))
joined.orderBy("device", "metric").show(truncate=False)
rows = {(r.device, r.metric) for r in joined.collect()}
n_summary = sum(1 for r in joined.collect() if r.metric == "batch")
assert ("m1", "temp") in rows and ("m1", "rpm") in rows and ("m2", "checked") in rows
assert n_summary == 3                      # terminate() fired once per partition

# 3) SQL path: an unregistered Python UDTF is NOT resolvable in LATERAL TABLE
try:
    spark.sql("SELECT * FROM machines, LATERAL TABLE(HealthExplode(device, temp_c, rpm)) h")
    print("LATERAL TABLE resolved (unexpected)")
except Exception as e:
    print("LATERAL TABLE without catalog registration ->", type(e).__name__,
          str(e)[:80])
print("Q5 OK: @udtf eval/terminate semantics; lateralJoin per-row; terminate fires once per partition; "
      "SQL TABLE() needs the function registered in the catalog")

## Q6 — PySpark Coding — Time-Weighted Averages (TWAP) over State Intervals `[HARD]`

**Scenario (Tesla telemetry / Uber machines):** sensors report **state changes**, not samples: `idle → run → idle`. "Average power" must be weighted by **how long each state lasted** — a naive `AVG(power)` treats a 5-second blip the same as a 40-minute run.

**Requirements**
1. Derive each interval's duration with `LEAD(ts) OVER (PARTITION BY machine ORDER BY ts)`; the last state's duration is unobserved — decide how to handle it (here: coalesce to 0 and disclose the censoring).
2. Compute `TWAP = Σ(power × minutes) / Σ(minutes)` per machine, plus `observed_minutes` so consumers know the coverage; show how different the naive row-average is.
3. Explain the general pattern: duration-weighting turns **state intervals** into **sample averages**; point out the same math powers VWAP, uptime %, and per-hour occupancy — and why you must **re-aggregate from raw intervals, never average averages**.

**Interview traps:** summing durations of the last state (unobserved!) inflates totals; mixing units (unix seconds vs minutes); averaging per-machine TWAPs to get a fleet TWAP (only correct with equal observed minutes).

In [0]:
# Solution Q6 — TWAP of state intervals via LEAD + duration weighting
states = spark.createDataFrame([
    ("mach1", dt.datetime(2026,11,24, 9, 0), "idle",  0.0),
    ("mach1", dt.datetime(2026,11,24, 9,10), "run", 100.0),
    ("mach1", dt.datetime(2026,11,24, 9,40), "idle",  0.0),
    ("mach1", dt.datetime(2026,11,24,10, 0), None,     None),   # observation window end
    ("mach2", dt.datetime(2026,11,24, 9, 0), "run",  80.0),
    ("mach2", dt.datetime(2026,11,24, 9,30), "idle",  0.0),
    ("mach2", dt.datetime(2026,11,24,10, 0), None,     None),
], "machine string, ts timestamp, state string, power double")

w6 = Window.partitionBy("machine").orderBy("ts")
durations = (states
    .withColumn("next_ts", F.lead("ts").over(w6))
    .withColumn("mins", F.coalesce(
        (F.unix_timestamp("next_ts") - F.unix_timestamp("ts")) / 60.0, F.lit(0.0))))  # censor last state
durations.orderBy("machine", "ts").select("machine", "ts", "state", "power", "mins").show()

twap = (durations.filter("mins > 0")
       .groupBy("machine")
       .agg((F.sum(F.col("power") * F.col("mins")) / F.sum("mins")).alias("twap_power"),
            F.sum("mins").alias("observed_minutes")))
twap.orderBy("machine").show()
t = {r.machine: (round(r.twap_power, 4), r.observed_minutes) for r in twap.collect()}
# mach1: 10min idle@0 + 30min run@100 + 20min idle@0 -> 3000/60 = 50.0 over 60 observed minutes
# mach2: 30min run@80 + 30min idle@0 -> 2400/60 = 40.0 over 60 minutes
assert abs(t["mach1"][0] - 50.0) < 1e-6 and abs(t["mach2"][0] - 40.0) < 1e-6, t
assert t["mach1"][1] == 60.0 and t["mach2"][1] == 60.0

naive = {r.machine: round(r.naive_avg, 2) for r in
         (durations.filter("mins > 0").groupBy("machine")
          .agg(F.avg("power").alias("naive_avg")).collect())}
print("naive row-average (misleading):", naive)
assert abs(naive["mach1"] - 100/3) < 0.05
print("Q6 OK: TWAP mach1=50.0 (naive avg 33.33!), mach2=40.0 — duration weighting via LEAD; "
      "last-state censoring handled and disclosed via observed_minutes")

## Q7 — Spark Optimization — Python Worker Memory Model & Arrow `[HARD]`

**Scenario (Databricks support escalation):** a pandas UDF job dies with `MemoryError` in the **Python worker**, but the executor JVM had plenty of free heap and the driver is fine. You must explain the process topology and tune the right thing.

**Requirements**
1. Whiteboard the memory topology of a PySpark executor: JVM heap (Spark memory fractions) **plus** a forked **python daemon/worker process per task** with its own RSS; `spark.executor.pyspark.memory` bounds the worker; Arrow batches serialize between JVM and Python — `spark.sql.execution.arrow.pyspark.enabled` and `spark.sql.execution.arrow.maxRecordsPerBatch` control the per-batch footprint.
2. Give the **OOM triage decision tree**: Python worker `MemoryError` (huge pandas frames — lower batch size / filter before collect_list / avoid `.collect()`), executor `OutOfMemoryError` (storage/shuffle fractions, spills), driver OOM (collect, broadcast of big results).
3. Quantify batch math: 10M rows × 100 bytes through a pandas UDF at `maxRecordsPerBatch=10,000` → how many batches, bytes per batch, and why 10× larger batches increase throughput until they OOM the worker (batch spill math below).

**Talk track only (config values are environment-managed on serverless — reads are guarded).**

In [0]:
# Solution Q7 — Python-worker memory model: guarded probes + batch-spill math + triage tree
def conf(name):
    try:
        return spark.conf.get(name)
    except Exception:
        return "NOT_SETTABLE_HERE (serverless-managed)"

for c in ["spark.sql.execution.arrow.pyspark.enabled",
          "spark.sql.execution.arrow.maxRecordsPerBatch",
          "spark.executor.pyspark.memory"]:
    print(f"{c:55s} = {conf(c)}")

# batch math: how much Python-side memory one Arrow batch needs
rows_total, bytes_per_row, batch_rows = 10_000_000, 100, 10_000
batches = rows_total / batch_rows
batch_bytes = batch_rows * bytes_per_row / 1e6
print(f"pandas UDF over {rows_total:,} rows: {batches:,.0f} batches × {batch_bytes:.1f} MB/batch")
print("bump maxRecordsPerBatch 10x -> 100 MB/batch: fewer round trips, but one batch "
      "now needs a contiguous 100 MB in the worker — the classic worker MemoryError")
assert batches == 1000 and abs(batch_bytes - 1.0) < 1e-9

TALK = [
  "Topology: each executor JVM forks python worker processes; the worker's RSS is NOT part of "
  "the JVM heap you sized — spark.executor.pyspark.memory is the dedicated budget.",
  "Data path: JVM rows -> Arrow batches (columnar, off-heap buffers) -> python worker -> pandas "
  "DataFrame -> your function -> Arrow back. Every hop copies once.",
  "Triage 1: MemoryError in the worker => shrink batches (maxRecordsPerBatch), project fewer "
  "columns BEFORE the UDF, avoid collect_list of huge groups (use applyInPandas per group).",
  "Triage 2: executor OutOfMemoryError => unified-memory fractions / shuffle spill, NOT the python "
  "knobs; check Spark UI for spilling stages first.",
  "Triage 3: driver OOM => collect() / toPandas() / broadcast of a huge result; move compute to "
  "executors, stream results with toPandas(batch_size=...).",
  "Rule of thumb: the worker needs ~2-3x the single-batch size for pandas temporaries; size "
  "maxRecordsPerBatch so batch_bytes * 3 < pyspark.memory.",
]
for line in TALK:
    print("- " + line)
print("Q7 OK: topology + triage tree + batch math (1,000 batches x 1.0 MB at 10k records/batch)")

## Q8 — Spark Deep — State Store Internals: HDFS-Backed vs RocksDB `[FAANG]`

**Scenario (Netflix real-time personalization / Uber streaming):** a stateful stream (`mapGroupsWithState`, stream-stream join, or dedup) runs 24/7. Every micro-batch must **checkpoint its state** so failures can resume exactly where they stopped. The default `HDFSBackedStateStoreProvider` writes a **full snapshot per version**; newer deployments switch to **RocksDB** state stores.

**Requirements**
1. Explain the two mechanisms: HDFS-backed = version N is a full snapshot dir written per micro-batch (delta files + periodic `.snapshot` consolidation); RocksDB = LSM-tree on local disk with **changelog checkpointing** — only the state **delta** ships to cloud storage per batch, plus compaction in the background.
2. Give the **cost model**: snapshot checkpoint IO per micro-batch ≈ O(total state size) vs changelog ≈ O(records changed). With 5M keys × 200B state and 1% churn per 30s micro-batch, compute both and the reduction.
3. Decision matrix: when RocksDB wins (large state > a few GB, high write rate, remote checkpoint bandwidth limits) vs when the default is fine (small state, simplicity, HDFSBacked’s simpler failure-recovery paths); mention metrics to watch (`stateStore.customMetric.*` — numKeys, commit-latency, changelog-bytes) and the config knobs (provider class, changelog checkpointing, block cache).

**Talk track + concrete simulation (no live stateful job needed).**

In [0]:
# Solution Q8 — state store checkpoint economics: snapshots vs changelogs (simulation)
keys, state_bytes, churn_pct, batch_seconds, batches = 5_000_000, 200, 0.01, 30, 500
state_gb = keys * state_bytes / 1e9
print(f"state: {keys:,} keys × {state_bytes}B = {state_gb:.1f} GB | micro-batch every {batch_seconds}s")

snapshot_io_gb = batches * state_gb                       # full snapshot every micro-batch
changelog_gb = batches * keys * churn_pct * state_bytes / 1e9   # only the changed keys
print(f"HDFS-backed snapshots : {snapshot_io_gb:,.0f} GB written to cloud storage over {batches} batches")
print(f"RocksDB changelog     : {changelog_gb:,.0f} GB written for the same window")
reduction = 1 - changelog_gb / snapshot_io_gb
print(f"reduction: {reduction*100:.0f}% — plus RocksDB keeps hot state on local disk (LSM) instead of heap")
assert 0.98 < reduction < 1.0

TALK = [
  "HDFSBackedStateStoreProvider: every micro-batch writes version-N files = delta + index, with a "
  "full .snapshot consolidation periodically; recovery reads the latest snapshot + replay deltas.",
  "RocksDB state store: state lives in an LSM-tree on the executor's local disk; per-batch cloud IO is "
  "the CHANGELOG (put/delete records) only; background compaction keeps read amplification flat.",
  "Changelog math: IO per micro-batch is O(changed keys) instead of O(total keys) — the 99% reduction above.",
  "RocksDB wins when: state > a few GB, high write churn, limited remote-fs bandwidth, or you need to "
  "avoid the periodic full-snapshot write stall.",
  "Default is fine when: state is small (KB-MB), ops simplicity matters, you want the simplest recovery.",
  "Watch: stateStore.customMetric.numKeys, .commitLatencyMs, .changelogBytes in the StreamingQueryListener "
  "progress JSON (Day 31 covered the listener contract).",
  "Knobs: spark.sql.streaming.stateStore.providerClass, rocksdb block cache + write-buffer sizes; "
  "changelog checkpointing is enabled at the checkpoint/stream level on supported deployments.",
]
for line in TALK:
    print("- " + line)
print("Q8 OK: snapshot vs changelog economics quantified (100% churn-free saving: ~99%); "
      "decision matrix + metrics + knobs on the whiteboard")

## Q9 — ETL — DLQ Replay with Per-Key FIFO Ordering `[HARD]`

**Scenario (Amazon payments ingestion):** a nightly batch lands ledger events for many accounts. A poison record (negative amount) fails the batch. Ops has two goals: (a) **don't let one account's poison block every other account** (head-of-line blocking), and (b) when the poison is fixed and **replayed**, per-account **ordering** must still hold — a stale event must never overwrite a newer one.

**Requirements**
1. Demonstrate strict global-FIFO replay: one poison event stops everything behind it; count the blocked events.
2. Switch to **per-key queues**: the poison parks only its own key's suffix (order preserved by parking the remainder of that key's sequence); other keys complete.
3. On replay, guard the MERGE with a **max-sequence condition** (`WHEN MATCHED AND s.seq > t.seq`): dedup the source per key first (source rows must match target keys at most once), and prove a stale replay is a no-op while a newer sequence wins.

**Interview traps:** MERGE source with duplicate keys errors (DELTA multiple-matches); parking only the poison event lets a later same-key event jump ahead (ordering violation); 'retry the whole batch' vs 'skip-and-continue' trade-off.

In [0]:
# Solution Q9 — head-of-line blocking demo, per-key queues, MERGE max-seq guard
events = [("acct_1",1,"dep",10), ("acct_1",2,"wdr",5), ("acct_1",3,"dep",7),
          ("acct_2",1,"dep",50), ("acct_2",2,"dep",-1),   # -1 => poison
          ("acct_1",4,"wdr",3), ("acct_2",3,"dep",2), ("acct_3",1,"dep",100)]

def process(ev):
    acct, seq, typ, amt = ev
    if amt < 0: raise ValueError("poison amount")
    return ev

# 1) strict global FIFO: one poison blocks everything behind it
ok_all, dlq_all = [], []
for ev in events:
    try:
        ok_all.append(process(ev))
    except ValueError:
        dlq_all.append(ev)
        break
blocked = len(events) - len(ok_all) - len(dlq_all)
print(f"strict global FIFO: processed={len(ok_all)}, parked={len(dlq_all)}, BLOCKED={blocked} events behind the poison")
assert blocked == 3

# 2) per-key queues: poison parks only its own key's suffix
per_key = defaultdict(deque)
for ev in events: per_key[ev[0]].append(ev)
out_ok, out_dlq = [], []
for acct, q in per_key.items():
    while q:
        ev = q.popleft()
        try:
            out_ok.append(process(ev))
        except ValueError:
            out_dlq.append(ev)
            out_dlq.extend(q); q.clear()    # park the remainder of THIS key to preserve order
            break
print("per-key isolation: ok =", [e[0]+":"+str(e[1]) for e in out_ok])
print("                    dlq =", [(e[0], e[1]) for e in out_dlq])
assert len(out_ok) == 6 and len(out_dlq) == 2
assert [e[1] for e in out_ok if e[0]=="acct_1"] == [1,2,3,4]   # per-key order preserved

# 3) replay guard: MERGE with max-seq condition (source deduped per key)
q9tbl = f"workspace.default.d34_q9_ledger_{uuid.uuid4().hex[:6]}"
spark.sql(f"CREATE TABLE {q9tbl} (acct STRING, seq INT, typ STRING, amt DOUBLE) USING DELTA")
def land(rows):
    src = spark.createDataFrame(rows, "acct string, seq int, typ string, amt double")
    (src.groupBy("acct").agg(F.max_by(F.struct("seq","typ","amt"), "seq").alias("latest"))
        .select("acct", "latest.seq", "latest.typ", "latest.amt").createOrReplaceTempView("src_q9"))
    spark.sql(f"""
      MERGE INTO {q9tbl} t USING src_q9 s ON t.acct = s.acct
      WHEN MATCHED AND s.seq > t.seq THEN UPDATE SET *
      WHEN NOT MATCHED THEN INSERT *
    """)
land([("acct_1",1,"dep",10), ("acct_1",2,"wdr",5)])
land([("acct_1",2,"wdr",5), ("acct_1",3,"dep",7)])   # older replay of seq2 + newer seq3
land([("acct_1",1,"dep",10)])                          # pure stale replay
res = spark.sql(f"SELECT acct, seq, typ FROM {q9tbl} ORDER BY seq").collect()
print("ledger after 3 lands (incl. stale replays):", [(r.acct, r.seq, r.typ) for r in res])
assert len(res) == 1 and res[0].seq == 3 and res[0].typ == "dep"
print("Q9 OK: per-key FIFO (no head-of-line blocking, 3 events unblocked) + "
      "MERGE max-seq guard makes stale replays no-ops; latest sequence wins")

## Q10 — ETL — Poison-Batch Bisection `[FAANG]`

**Scenario (Stripe / Airbnb backfill):** a 1,000-record batch fails on ONE malformed record. You cannot see which — the exception has no row context. Re-running record-by-record is 1,000 replays (slow, expensive, re-triggers side effects). Find the poison record in **O(log n)** replays.

**Requirements**
1. Implement **deterministic batch bisection**: try the full batch once; on failure, binary-search on row ranges — run `[lo, mid)`, if green the poison is in `[mid, hi)`, else recurse left. Stop at a single row; log the probe count.
2. **Quarantine** the isolated record (to the DLQ with the exception attached) and rerun the full batch clean; state the invariants: determinism (same input + code ⇒ same failure), probes ≤ log2(n)+1.
3. Discuss why this beats row-replay (cost math: probes vs n), the failure modes that BREAK bisection (nondeterministic errors — flaky network; errors only when records are processed TOGETHER — cross-record contention), and where you'd automate it (retry wrapper that bisects after the 2nd identical failure).

**Interview traps:** assuming bisection works for flaky errors (it requires determinism); forgetting the +1 initial full-batch probe; off-by-one on the half-open range `[lo, mid)`.

In [0]:
# Solution Q10 — bisect the failing batch to the exact poison record
random.seed(4)
batch = [{"rid": i, "payload": "ok" if i != 731 else "POISON"} for i in range(1000)]

def run_batch(rows):
    for r in rows:
        if r["payload"] == "POISON": raise RuntimeError("batch failed")
    return "ok"

def bisect_poison(rows):
    lo, hi, probes = 0, len(rows), 0
    probes += 1                       # probe 0: the full batch
    try:
        run_batch(rows); return None, probes
    except RuntimeError:
        pass
    while hi - lo > 1:                # half-open [lo, hi)
        mid = (lo + hi) // 2; probes += 1
        try:
            run_batch(rows[lo:mid]); lo = mid   # left green -> poison is right
        except RuntimeError:
            hi = mid                             # poison is left
    return lo, probes

idx, probes = bisect_poison(batch)
print(f"poison isolated: rid={batch[idx]['rid']} in {probes} probes vs {len(batch)} row-replays")
assert batch[idx]["rid"] == 731 and probes <= 11    # 1 + ceil(log2(1000)) = 1+10
clean = [r for i, r in enumerate(batch) if i != idx]   # quarantine rid 731 to DLQ
assert run_batch(clean) == "ok"
print(f"after quarantine: full rerun green with {len(clean)} rows")
print(f"cost math: {probes} probes vs {len(batch)} single-record replays = {len(batch)//probes}x cheaper")
print("Q10 OK: deterministic bisection finds rid 731 in 11 probes; "
      "requires deterministic failures; half-open ranges avoid off-by-one")

## Q11 — ETL — Delta Row-Level Lineage (Row Tracking) `[FAANG]`

**Scenario (Databricks 2025 feature / audited pipelines):** compliance asks "which **rows** in the serving table were written by which **commit**, and can you prove a row's identity survives updates?" Table-level history (DESCRIBE HISTORY) answers file/version questions — row-level lineage answers **row** questions: incremental maintenance, targeted erasure, audit joins from a fact row back to the commit that produced it.

**Requirements**
1. Enable row tracking: `ALTER TABLE ... SET TBLPROPERTIES ('delta.enableRowTracking' = true)` (or `delta.enableRowTracking` at creation) — Delta assigns every row an immutable **row_id** plus **row_commit_version**.
2. Read them via the `_metadata` struct (`_metadata.row_id`, `_metadata.row_commit_version`); verify **row_ids are stable across an UPDATE for untouched rows** and that the updated row carries the new commit version.
3. Whiteboard the use cases: per-row incremental pipelines (only rows with new commit versions flow downstream — cheaper than full CDF reads for some shapes), row→commit audit joins, and the caveats — full rewrites (OPTIMIZE / rewrite in place) can reassign ids, so treat row_id as **stable within the table's write history, not forever**; enabling it adds stored commit metadata.

**Interview traps:** row_id vs row_commit_version confusion; expecting `_metadata` columns in the schema listing (they're hidden, read via `_metadata.*`); asserting absolute version numbers (serverless auto-ops interleave).

In [0]:
# Solution Q11 — enable row tracking, prove row_id stability across an UPDATE
q11tbl = f"workspace.default.d34_q11_rowtrack_{uuid.uuid4().hex[:6]}"
spark.sql(f"CREATE TABLE {q11tbl} (id INT, v STRING) USING DELTA")
spark.sql(f"ALTER TABLE {q11tbl} SET TBLPROPERTIES ('delta.enableRowTracking' = true)")
spark.sql(f"INSERT INTO {q11tbl} VALUES (1,'a'), (2,'b'), (3,'c')")

# 1) hidden columns: NOT in schema.fields — read them via _metadata
visible = [c.name for c in spark.table(q11tbl).schema.fields]
print("visible schema columns:", visible, "(row_id / row_commit_version are hidden)")
probe = spark.sql(f"SELECT id, v, _metadata.row_id AS row_id, _metadata.row_commit_version AS rcv FROM {q11tbl} ORDER BY id")
probe.show()

# 2) UPDATE one row: untouched row_ids must NOT move; the updated row bumps rcv
before = {r.id: (r.row_id, r.rcv) for r in probe.collect()}
spark.sql(f"UPDATE {q11tbl} SET v = 'b2' WHERE id = 2")
after = {r.id: (r.row_id, r.rcv) for r in spark.sql(
    f"SELECT id, _metadata.row_id AS row_id, _metadata.row_commit_version AS rcv FROM {q11tbl}").collect()}
print("before:", before)
print("after :", after)
assert before[1][0] == after[1][0] and before[3][0] == after[3][0], "untouched row_ids changed!"
assert after[2][0] == before[2][0], "row_id should stay stable for the updated row"
assert after[2][1] > before[2][1], "updated row should carry the newer commit version"

# 3) audit use case: join rows back to the commit that produced them (version-relative, never absolute)
hist = spark.sql(f"DESCRIBE HISTORY {q11tbl}").select("version", "operation").collect()
print("history (newest first):", [(h.version, h.operation) for h in hist])
commits = spark.sql(f"""
  SELECT t.id, t._metadata.row_id AS row_id, h.operation AS commit_op
  FROM {q11tbl} t
  LEFT JOIN (SELECT version, operation FROM (DESCRIBE HISTORY {q11tbl})) h
    ON t._metadata.row_commit_version = h.version
  ORDER BY t.id
""")
commits.show(truncate=False)
print("Q11 OK: row_id stable across UPDATE (untouched rows unchanged, updated row re-committed); "
      "row -> commit audit join works; ids are stable within the write history, not across rewrites")

## Q12 — Data Modeling — Non-Events: Expected-vs-Actual Coverage Facts `[HARD]`

**Scenario (Walmart / Flipkart store analytics):** "store X made no sales for two days" is a real, answerable fact — but your fact table only contains events that **happened**. Non-events (no sale, no show, sensor dark) are invisible to `SUM(amt)` and even to `COUNT(*)`.

**Requirements**
1. Build the **expected-activity spine**: `open stores × calendar dates` (only days since each store's open date — not before). This is the Kimball trick: facts of absence need a **grain anchor that exists even when the event doesn't**.
2. LEFT JOIN the sales fact; `COALESCE(amt, 0)` + a `had_sale` flag; aggregate into a coverage report — expected_days / days_with_sales / **zero_days become countable facts**.
3. Explain when to persist the zero rows physically (a **coverage fact table** — small, dense, queryable) vs computing the spine on the fly; relate to factless fact tables (Day 4) and time-series gap-fill (different problem: filling **values**, not creating **rows for absence**); mention no-show scheduling (expected appointments × arrivals) as the same pattern.

**Interview traps:** building the spine from the fact's own dates (circular — zeros stay invisible); including dates before a store opened; inner join by accident; zero-amount sales ≠ no sale (COALESCE ordering matters).

In [0]:
# Solution Q12 — expected-vs-actual coverage spine: non-events become countable facts
stores = spark.createDataFrame([("s1","2026-11-20"), ("s2","2026-11-21"), ("s3","2026-11-20")],
                               "store string, open_date string").withColumn("open_date", F.to_date("open_date"))
dates = spark.createDataFrame([("2026-11-20",), ("2026-11-21",), ("2026-11-22",), ("2026-11-23",)], "d string") \
             .select(F.to_date("d").alias("date"))
sales = spark.createDataFrame([("s1","2026-11-20",120.0), ("s1","2026-11-21",80.0),
                               ("s2","2026-11-21",60.0), ("s3","2026-11-23",99.0)],
                               "store string, date string, amt double").withColumn("date", F.to_date("date"))

# 1) spine = open store × date, only on/after open date (grain anchor for absence)
spine = stores.join(dates, F.col("date") >= F.col("open_date")).select("store", "date")

# 2) LEFT JOIN + COALESCE: the zeros become real rows
cov = (spine.join(sales, ["store", "date"], "left")
       .withColumn("amt", F.coalesce("amt", F.lit(0.0)))
       .withColumn("had_sale", F.col("amt") > 0))
cov.orderBy("store", "date").show()

# 3) coverage report: zero_days are now COUNTABLE facts
report = cov.groupBy("store").agg(
    F.count("*").alias("expected_days"),
    F.sum(F.when(F.col("had_sale"), 1).otherwise(0)).alias("days_with_sales"),
    F.sum(F.when(~F.col("had_sale"), 1).otherwise(0)).alias("zero_days"))
report.orderBy("store").show()
rep = {r.store: (r.expected_days, r.days_with_sales, r.zero_days) for r in report.collect()}
# s1 open 11/20: 4 days, 2 sales -> 2 zero. s2 open 11/21: 3 days, 1 sale -> 2 zero.
# s3 open 11/20: 4 days, 1 sale -> 3 zero.
assert rep == {"s1": (4,2,2), "s2": (3,1,2), "s3": (4,1,3)}, rep
print("Q12 OK: spine {s1:(4d,2 sale,2 zero), s2:(3,1,2), s3:(4,1,3)} — "
      "absence-of-sales is now a queryable fact; fact-only tables would have shown nothing")

## Q13 — Data Modeling — Heterogeneous Multi-Channel Event Union `[FAANG]`

**Scenario (Meta / Adobe analytics):** events arrive from **web**, **app**, and **POS** channels with different columns (web/app carry engagement seconds; POS carries basket value). The dashboard team wants ONE fact table to query, not three joins.

**Requirements**
1. Decide the grain: **one row per event regardless of channel** — then union the three sources with `unionByName` (column-order-safe) into a single fact with a `chan` discriminator and **sparse measures** (NULL where the measure doesn't apply).
2. Build a **measure applicability matrix** (which measures are meaningful per channel) — the documentation artifact that prevents nonsense like 'POS engagement seconds' from ever being aggregated; demonstrate the sparsity with `count(measure)` vs `count(*)`.
3. Whiteboard the alternative designs and trade-offs: **unioned one-fact** (one table, sparse columns, wide schema evolution) vs **per-channel facts** (dense, per-channel SLOs, but cross-channel analytics needs UNION at query time) vs **supernova/common-core** split (shared core fact + channel satellites). State when each wins and the schema-evolution story for each.

**Interview traps:** positional `union` (silent column-order corruption) vs `unionByName`; summing a sparse measure across channels where it's undefined; claiming one grain then unioning facts with different grains.

In [0]:
# Solution Q13 — unionByName one-fact pattern + applicability matrix
web = spark.createDataFrame([("e1","web","u1","view_page",60), ("e2","web","u2","view_page",45)],
      "event_id string, chan string, uid string, action string, seconds int")
app = spark.createDataFrame([("e3","app","u1","push_open",1), ("e4","app","u1","purchase",None)],
      "event_id string, chan string, uid string, action string, seconds int")
pos = spark.createDataFrame([("e5","pos","cust_9","checkout",None)],
      "event_id string, chan string, uid string, action string, seconds int")

# 1) one grain — one row per event regardless of channel; unionByName is position-safe
uni = web.unionByName(app).unionByName(pos)
uni.orderBy("event_id").show()

# 2) sparsity made explicit: count(measure) vs count(*)
piv = uni.groupBy("chan").agg(F.count("*").alias("events"),
    F.count("seconds").alias("engagement_measure_count"),
    F.sum(F.coalesce("seconds", F.lit(0))).alias("sparse_sum"))
piv.orderBy("chan").show()
p = {r.chan: (r.events, r.engagement_measure_count) for r in piv.collect()}
assert p == {"app": (2,1), "pos": (1,0), "web": (2,2)}, p

# 3) the applicability matrix: the artifact that stops nonsense aggregates
matrix = spark.createDataFrame([
    ("web", "seconds", True),      ("web", "basket_value", False),
    ("app", "seconds", True),      ("app", "basket_value", False),
    ("pos", "seconds", False),     ("pos", "basket_value", True),
], "chan string, measure string, applies boolean")
matrix.orderBy("chan", "measure").show()
TALK = [
  "One-fact wins when: cross-channel funnels dominate, one governance/SLO surface, moderate sparsity.",
  "Per-channel facts win when: channels have different grains/SLAs owners, dense per-channel measures, "
  "independent retention; cross-channel queries pay a UNION at query time.",
  "Supernova/common-core: shared core columns (event_id, ts, actor, chan) + per-channel satellite "
  "tables for channel-specific measures — the compromise when both matter.",
  "Schema evolution: one-fact needs unionByName + additive-only changes (NULL backfill); "
  "per-channel tables evolve independently but the union query must keep names aligned.",
]
for line in TALK:
    print("- " + line)
print("Q13 OK: 5 events, one fact; seconds applies 4/5 rows (web 2/2, app 1/2, pos 0/1); "
      "unionByName is the position-safe union; matrix documents applicability")

## Q14 — Snowflake — SNOWFLAKE.ML Functions (FORECAST / ANOMALY_DETECTION) `[FAANG]`

**Scenario (retail demand planning on Snowflake):** the team wants next-week revenue forecasts and anomaly flags on a metrics table — **inside Snowflake, no data egress, no external ML service**. Snowflake's built-in `SNOWFLAKE.ML` schema ships `FORECAST`, `ANOMALY_DETECTION`, `TOP_INSIGHTS`, and `CLASSIFICATION` as SQL-callable classes.

**Requirements**
1. Show the actual call shape: `CALL SNOWFLAKE.ML.FORECAST(INPUT => ..., TIMESTAMP_COLNAME => ..., VALUE_COLNAME => ..., CONFIG_OBJECT => {'prediction_interval': 0.95})` then `CALL ...!FORECAST(FORECASTING_PERIODS => 7)` — explain the **two-step session pattern** (CREATE a forecaster instance binding data, then invoke its `!METHOD`), and that SNOWFLAKE.ML functions consume DB credits (compute via warehouse) while keeping data in-place.
2. Same pattern for `ANOMALY_DETECTION(...)!DETECT` with a contamination threshold; explain why anomalies come back as a series-level report and how you'd persist results to a table (CTAS from the result relation).
3. Compare against Databricks-side alternatives (SQL AI functions `ai_forecast` / `ai_analyze` — LLM-backed, one-shot functions vs class-instantiation), and explain when built-in ML functions win (no egress, SQL-native, governed) vs when a real pipeline (feature store + retraining) is required (nonlinearity, exogenous regressors, production SLAs).

**Interview traps:** forgetting the `!` instance-method call; thinking SNOWFLAKE.ML is free (it runs on your warehouse); mixing up forecasting periods (steps) vs interval (confidence).

In [0]:
# Solution Q14 — talk track + shape validation (Snowflake SQL is shown as strings: not executable here)
FORECAST_DDL = """
CREATE SNOWFLAKE.ML.FORECAST demand_forecaster(
  INPUT => SYSTEM$REFERENCE('VIEW', 'metrics.v_revenue_daily'),
  TIMESTAMP_COLNAME => 'day',
  VALUE_COLNAME  => 'revenue',
  CONFIG_OBJECT => {'prediction_interval': 0.95, 'seasonality': 'auto'}
);
CALL demand_forecaster!FORECAST(FORECASTING_PERIODS => 7);   -- result: TS, FORECAST, LOWER_BOUND, UPPER_BOUND
"""
ANOMALY_DDL = """
CREATE SNOWFLAKE.ML.ANOMALY_DETECTION order_anom(
  INPUT => SYSTEM$REFERENCE('VIEW', 'metrics.v_orders_hourly'),
  SERIES_COLNAME => 'region',
  TIMESTAMP_COLNAME => 'ts',
  VALUE_COLNAME => 'orders',
  CONFIG_OBJECT => {'on_error': 'skip'}
);
CALL order_anom!DETECT(ANCHOR_TIMESTAMP => NULL,  -- NULL = detect across the whole series
  CONFIG_OBJECT => {'containment': 0.01});        -- ~1% expected anomalies
"""
PERSIST = """
CREATE TABLE metrics.anomaly_flags AS            -- persist the result relation for dashboards
SELECT ts, region, orders, is_anomaly, anomaly_score
FROM TABLE(RESULT_SCAN(-1)) ORDER BY ts;
"""
print(FORECAST_DDL); print(ANOMALY_DDL); print(PERSIST)
TALK = [
  "Two-step pattern: CREATE binds the model instance to data; the !METHOD call runs inference. "
  "The instance is a schema object with privileges, not a table function.",
  "Multi-series forecasting: add SERIES_COLNAME -> one model per series, one result set.",
  "Cost: SNOWFLAKE.ML runs inside your warehouse (credits/hour) — data never leaves the account; "
  "contrast with Snowpark container ML or external services that pay egress + latency.",
  "Databricks counterpart: ai_forecast / ai_analyze SQL functions are LLM-backed one-shot calls; "
  "Agent Bricks / real pipelines win once you need retraining, feature stores, or nonlinearity.",
  "Governing the output: CTAS from RESULT_SCAN(-1) or the !CALL result lands the forecast in a "
  "versioned table so BI reads a stable relation instead of a session-scoped result.",
]
for line in TALK:
    print("- " + line)
# shape sanity: the call syntax strings parse and contain the required keywords
for kw in ["SNOWFLAKE.ML.FORECAST", "!FORECAST", "ANOMALY_DETECTION", "!DETECT", "RESULT_SCAN"]:
    assert kw in FORECAST_DDL + ANOMALY_DDL + PERSIST, kw
print("Q14 OK: two-step CREATE + !CALL pattern, contamination config, persistence via CTAS, "
      "cost/governance trade-offs on the whiteboard")

## Q15 — Snowflake — Marketplace Listings & Data Monetization `[HARD]`

**Scenario (data provider on Snowflake Marketplace):** you publish a 200 GB geospatial POI dataset as a **standard listing**. Finance asks: what does it cost US to host it, what do consumers pay, and when would a **personalized listing** (auto-fulfill all consumer requests) change the calculus?

**Requirements**
1. Economics of a standard listing: the **provider's remote storage cost** (hosted data at ~$23/TB-month compressed) vs **consumer compute** (queries scan the listing; consumer pays their own warehouse credits per TB scanned — roughly 2 credits/TB at ~$3/credit). Compute both for the scenario.
2. Explain the listing types: **standard** (provider approves each request; free, limited, or per-terabyte pricing), **personalized** (auto-fulfill + provider runs code on consumer data), and **data exchange** (private, invite-only); say which one you'd pick for a paid POI feed and why.

**Interview traps:** thinking the provider pays for consumer queries (consumers pay their own compute); confusing listing pricing (per-TB or flat monthly) with the underlying storage cost; assuming auto-fulfill exists on standard listings.

In [0]:
# Solution Q15 — monetization math: provider storage vs consumer compute
GB = 200.0
price_per_tb_month = 23.0            # provider-side hosted storage, ~$/TB-month
avg_query_tb = 0.005                 # 5% of the listing touched per consumer query
queries_per_month = 4000
cr_per_tb = 2.0                      # warehouse credits consumed per TB scanned
cr_price = 3.0                       # ~$ per credit

provider_storage = GB / 1000 * price_per_tb_month
consumer_compute = queries_per_month * avg_query_tb * cr_per_tb * cr_price
print(f"listing {GB:.0f} GB: provider storage ≈ ${provider_storage:.1f}/mo")
print(f"consumer compute ≈ ${consumer_compute:.0f}/mo "
      f"({queries_per_month:,} queries × {avg_query_tb} TB × {cr_per_tb} cr/TB × ${cr_price}/cr)")
assert provider_storage < 5 and consumer_compute > 100

TALK = [
  "Asymmetry: the provider's marginal cost is storage only (~$4.6/mo for 200 GB); consumers pay "
  "their own compute on every scan — the listing can be priced well below consumer value.",
  "Pricing modes: free, per-TB, or flat monthly on the listing — decoupled from the storage cost.",
  "Standard listing: each consumer request needs provider approval — fine at low volume, friction at scale.",
  "Personalized listing: auto-fulfills every request AND lets the provider attach app logic "
  "(native apps) — right when you outgrow approvals or need consumer-side processing.",
  "Data exchange: private catalog for named partners — governance-first, no public discovery.",
  "For a paid POI feed: start standard + per-TB pricing (test demand), move to personalized once "
  "request volume makes manual approval the bottleneck.",
]
for line in TALK:
    print("- " + line)
print("Q15 OK: provider $4.6/mo vs consumer ~$120/mo compute for the scenario; "
      "listing-type decision matrix (standard vs personalized vs exchange) rehearsed")

## Q16 — Monitoring — Metrics Cardinality Explosion `[HARD]`

**Scenario (Netflix / Uber observability):** a 6-label metric (`cluster, service, endpoint, status_code, dc, shard`) quietly generates **millions of time series** in the backend — ingest cost and query latency blow past budget. You must quantify the blast radius and design the pre-aggregation ladder before shipping the metric.

**Requirements**
1. Compute the **label-space product** for the scenario dims (cluster×service×endpoint×status_code×dc) and the realistic **active series** at ~1% sparsity; then the reduction from folding a high-cardinality low-value label (`endpoint` → its parent `service`).
2. Build the cost model: bytes/day = active series × 1,440 scrapes (1-min resolution) × 16 B/sample; compare full vs folded ingest in GB/day.
3. State the **pre-aggregation rules**: aggregate before storage where the business question allows (count/sum — safe); **never pre-average percentiles** — merge histogram **buckets**, not averages (show the bucket-merge for a p50 across two hosts); label hygiene (bounded enums, no user-ids/emails as labels); recording rules that pre-roll rare queries; and the blast-radius guardrail (alert when active-series growth rate exceeds X%/week — the leading indicator before cost).

**Interview traps:** averaging percentiles across hosts (mathematically wrong); treating product-space as active-series (sparsity ~1–3% in real fleets); allowing unbounded labels (user_id, request_id) that make series count ~ row count.

In [0]:
# Solution Q16 — cardinality math, fold economics, histogram-merge rule
# 1) label-space product vs realistic active series
dims = {"cluster": 6, "service": 40, "endpoint": 500, "status_code": 8, "dc": 4}
series_full = math.prod(dims.values())
active_full = int(series_full * 0.01)            # ~1% sparsity in a real fleet
print(f"full label space: {series_full:,} product series -> ~{active_full:,} active at 1% sparsity")

# fold endpoint (high-cardinality, low analytical value) into service
dims_folded = {"cluster": 6, "service": 40, "status_code": 8, "dc": 4}
series_folded = math.prod(dims_folded.values())
active_folded = int(series_folded * 0.05)        # slightly higher activity share after folding
print(f"fold endpoint: {series_folded:,} product series -> ~{active_folded:,} active")
assert active_folded < active_full * 0.1

# 2) ingest cost: 1-min resolution, 16 B/sample
bytes_per_day = lambda active: active * 1440 * 16
gb_full, gb_folded = bytes_per_day(active_full)/1e9, bytes_per_day(active_folded)/1e9
print(f"ingest at 1-min resolution: full = {gb_full:.1f} GB/day vs folded = {gb_folded:.2f} GB/day")

# 3) the percentile rule: merge BUCKETS, never averages
b_host1 = [10, 30, 40, 20]     # latency-bucket counts, host 1
b_host2 = [50, 30, 15, 5]      # host 2
merged = [a + b for a, b in zip(b_host1, b_host2)]
cum = list(itertools.accumulate(merged))
p50_bucket = next(i for i, c in enumerate(cum) if c >= sum(merged) / 2)
print(f"bucket-merged p50 falls in bucket index {p50_bucket} (0 = fastest)")
print("naive per-host p50 average is NOT the fleet p50 — averages of quantiles are not quantiles")
TALK = [
  "Aggregate before store when the question allows: count/sum/gauge-max are safe to pre-roll.",
  "Percentiles: store histogram buckets (cumulative or per-bucket counts), merge buckets, then "
  "interpolate — never average p50s.",
  "Label hygiene: bounded enums only; user_id/email/request_id as labels => series count ~ row count.",
  "Recording rules: pre-roll the queries dashboards actually run; raw series behind a sample flag.",
  "Guardrail: alert on active-series growth rate (e.g., >10%/week) — cost is the lagging indicator.",
]
for line in TALK:
    print("- " + line)
print("Q16 OK: product space 3.84M -> ~38K active; fold cuts ~95%+; ingest math + bucket-merge rule validated")

## Q17 — Monitoring — MTTR Analytics on system.lakeflow.job_run_timeline `[HARD]`

**Scenario (platform SRE):** leadership wants **MTTR** (mean time-to-restore) per job for last 30 days, straight from the **system tables** — no extra instrumentation. Runs land in `system.lakeflow.job_run_timeline` with `run_type` separating `ORIGINAL_RUN` from `REPAIR_RUN` (repair stats must not pollute failure counts), and `result_state` marking outcomes.

**Requirements**
1. Pair each failure with the next successful run **per job** via `LEAD` — but the **window-then-filter trap**: if you `WHERE failed = 1` *inside* the same SELECT as the window, `LEAD` sees only failed rows and the next-success never materializes. Compute the window over ALL runs, then filter.
2. Define the two MTTRs and compute both: **last-failure→green** (repair time) and **first-failure→green** (outage duration) — consecutive failures make them differ; cross-check one against a Python brute force.
3. Query the **live** system table (30-day window, repairs excluded from the failure census) guarded for missing permission — on a fresh workspace it may legitimately return zero cycles; the synthetic timeline always demonstrates the math. Close with the SRE loop: MTTR Pareto by job, link `job_id` to `system.lakeflow.jobs` for owners/alert channels, and set the SLI the alerting stack uses.

**Interview traps:** LEAD-after-filter (the trap above); averaging outage durations with unequal multiples; double-counting a failure both as 'failure' and 'restored' when consecutive failures run >2.

In [0]:
# Solution Q17 — MTTR: live system-table query (guarded) + synthetic timeline + brute-force check
try:
    live = spark.sql("""
      WITH r AS (
        SELECT job_id, period_end_time, result_state FROM system.lakeflow.job_run_timeline
        WHERE period_start_time >= current_timestamp() - INTERVAL 30 DAYS
          AND result_state IS NOT NULL
      ), windowed AS (
        SELECT *, CASE WHEN result_state = 'SUCCEEDED' THEN 0 ELSE 1 END AS failed,
               LEAD(period_end_time) OVER (PARTITION BY job_id ORDER BY period_end_time) AS ok_time,
               LEAD(CASE WHEN result_state = 'SUCCEEDED' THEN 0 ELSE 1 END)
                 OVER (PARTITION BY job_id ORDER BY period_end_time) AS next_failed
        FROM r
      ), pairs AS (
        SELECT job_id, period_end_time AS fail_time, ok_time, next_failed
        FROM windowed WHERE failed = 1            -- filter AFTER the window (the trap)
      )
      SELECT count(*) AS failure_events,
             round(avg(unix_timestamp(ok_time) - unix_timestamp(fail_time))/3600.0, 2) AS mttr_hours_lastfail
      FROM pairs WHERE next_failed = 0 AND ok_time IS NOT NULL
    """)
    live.show()
    print("live path OK (may be 0 cycles on a quiet workspace — that is a valid answer, not a bug)")
except Exception as e:
    print("live system table unavailable:", type(e).__name__, str(e)[:100], "— run the synthetic half")

# synthetic timeline: hourly runs, failure blocks, a repair run, and an INTERNAL_ERROR
runs = []
t0 = dt.datetime(2026, 11, 20, 0, 0)
status = ["SUCCEEDED"]*23 + ["FAILED", "FAILED", "SUCCEEDED"]*2 + ["SUCCEEDED"]*10 \
        + ["INTERNAL_ERROR", "SUCCEEDED"] + ["SUCCEEDED"]*5
for i, s in enumerate(status):
    runs.append((501, 9000+i, t0 + dt.timedelta(hours=i), s))
spark.createDataFrame(runs, "job_id int, run_id int, period_end_time timestamp, result_state string") \
     .createOrReplaceTempView("sim_runs")
sim = spark.sql("""
  WITH marked AS (
    SELECT *, CASE WHEN result_state = 'SUCCEEDED' THEN 0 ELSE 1 END AS failed FROM sim_runs
  ), windowed AS (
    SELECT *, LEAD(period_end_time) OVER (PARTITION BY job_id ORDER BY period_end_time) AS ok_time,
              LEAD(failed)      OVER (PARTITION BY job_id ORDER BY period_end_time) AS next_failed
    FROM marked
  ), pairs AS (
    SELECT job_id, period_end_time AS fail_time, ok_time, next_failed
    FROM windowed WHERE failed = 1
  )
  SELECT count(*) AS failure_events,
         round(avg(unix_timestamp(ok_time) - unix_timestamp(fail_time))/3600.0, 2) AS mttr_hours
  FROM pairs WHERE next_failed = 0 AND ok_time IS NOT NULL GROUP BY job_id
""")
sim.show()
r = sim.collect()[0]
print("sim failure_events:", r.failure_events, "MTTR (last-fail -> green):", r.mttr_hours, "hours")

# brute force cross-check: both definitions
cycles_last, cycles_first = [], []
i = 0
while i < len(status):
    if status[i] != "SUCCEEDED":
        j = i
        while j < len(status) and status[j] != "SUCCEEDED": j += 1
        if j < len(status):
            cycles_first.append(j - i)
            cycles_last.append(1)
        i = j
    else:
        i += 1
assert r.failure_events == len(cycles_last)
assert abs(r.mttr_hours - sum(cycles_last)/len(cycles_last)) < 0.01
print(f"brute force: {len(cycles_first)} outage cycles, outage duration avg "
      f"{sum(cycles_first)/len(cycles_first):.2f}h, repair (last-fail) avg "
      f"{sum(cycles_last)/len(cycles_last):.2f}h")
print("Q17 OK: window-then-filter pairing == brute force; MTTR-repair 1.0h vs outage-duration 1.67h; "
      "live query guarded for permissions/quiet workspaces")

## Q18 — Monitoring — Usage-Weighted Data Product ROI Scorecard `[FAANG]`

**Scenario (Netflix data platform governance):** the platform runs 60+ gold tables with real storage+compute cost but wildly uneven usage. Before the quarterly review, rank every data product by **usage-weighted ROI** and propose a KEEP / REVIEW / ARCHIVE tier per table — defensible from **query logs**, not opinion.

**Requirements**
1. Build the scorecard: monthly storage + compute cost from billing, weekly distinct users and query counts from query history (Databricks: `system.access_taxonomy`/`system.query.history`; Snowflake: `ACCOUNT_USAGE.QUERY_HISTORY` — say where the numbers come from), downstream-dependency count from lineage.
2. Compute **cost-per-user** and **cost-per-1k-queries** with a zero-usage guard (NULL, not division by zero); assign tiers — e.g., queries ≥ 1,000/month KEEP, 1–999 REVIEW, 0 queries ARCHIVE_CANDIDATE (with a lineage check: no downstream before archive).
3. Explain how this differs from Day 33's composite health score (freshness+quality+usage for *operations*) — this is a **finance-facing portfolio view** — and how the same ACCESS_HISTORY/audit-query log powers governance forensics (Day 28). Discuss the human loop: owners get the REVIEW list before anything is archived (metadata 'last accessed' from logs, not table comments).

**Interview traps:** cost-per-user division by zero (0 users); counting scans by the table owner as 'usage'; trusting table comments for last-access; archive without lineage check breaks downstream.

In [0]:
# Solution Q18 — ROI scorecard: KEEP / REVIEW / ARCHIVE_CANDIDATE tiers from usage logs
products = spark.createDataFrame([
    # table_name, monthly_storage_cost, monthly_compute_cost, weekly_distinct_users, weekly_queries, downstream
    ("fct_orders",       40.0, 800.0, 42, 5200, 9),
    ("fct_clickstream", 120.0, 450.0,  3,   90, 4),
    ("dim_customers",     8.0,  40.0, 25, 1800, 12),
    ("mart_marketing_v2", 15.0,  90.0,  0,    0, 0),   # orphaned: no users, no queries, no lineage
    ("fct_payments",     60.0, 300.0, 18,  950, 6),
], "tbl string, storage double, compute double, users int, queries int, downstream int")

# 1-2) scorecard with zero-usage guards (NULL, never /0)
roi = (products
    .withColumn("total_cost", F.col("storage") + F.col("compute"))
    .withColumn("cost_per_user", F.round(F.when(F.col("users") > 0, F.col("total_cost") / F.col("users")), 2))
    .withColumn("cost_per_1k_queries",
        F.round(F.when(F.col("queries") > 0, F.col("total_cost") / (F.col("queries") / 1000.0)), 2))
    .withColumn("tier", F.when(F.col("queries") >= 1000, "KEEP")
                          .when((F.col("queries") > 0) & (F.col("queries") < 1000), "REVIEW")
                          .otherwise("ARCHIVE_CANDIDATE")))
roi.orderBy(F.col("total_cost").desc()).show()

tiers = {r.tbl: r.tier for r in roi.collect()}
assert tiers == {"fct_orders": "KEEP", "dim_customers": "KEEP",
                 "fct_clickstream": "REVIEW", "fct_payments": "REVIEW",
                 "mart_marketing_v2": "ARCHIVE_CANDIDATE"}, tiers
cpu_ = {r.tbl: r.cost_per_user for r in roi.collect()}
assert cpu_["mart_marketing_v2"] is None and cpu_["fct_orders"] == 20.0

# 3) archive safety: lineage check gates the ARCHIVE_CANDIDATE
archive_safe = roi.where("tier = 'ARCHIVE_CANDIDATE' AND downstream = 0").select("tbl").collect()
assert [r.tbl for r in archive_safe] == ["mart_marketing_v2"]
TALK = [
  "Sources: costs from billing/system.billing.usage; users+queries from query history "
  "(system.access.* / QUERY_HISTORY on Snowflake); downstream from lineage — never from comments.",
  "Zero-usage guard: NULL cost-per-user, not a crash; and 'usage' excludes the owner's own scans.",
  "vs Day 33 health score: that one is operational (freshness+quality+usage for on-call); "
  "this one is the finance-facing portfolio view for the quarterly review.",
  "Human loop: REVIEW owners get the cost-per-user numbers first; ARCHIVE requires the lineage "
  "check (downstream = 0) plus an owner sign-off window before drop.",
]
for line in TALK:
    print("- " + line)
print("Q18 OK: tiers KEEP(fct_orders, dim_customers) / REVIEW(fct_clickstream, fct_payments) / "
      "ARCHIVE_CANDIDATE(mart_marketing_v2, lineage-gated); zero-usage returns NULL not /0")

# 📋 Day 34 — Scorecard

Attempt each question **before** opening its solution. Score 1–5 per question; target ≥ 4 on every row before moving to Day 35.

| # | Topic | Tag | Self-score (1–5) | Notes |
| --- | --- | --- | --- | --- |
| Q1 | Next-available-slot scheduling (spine + anti-join) | [FAANG] |  |  |
| Q2 | Repeatable sampling: TABLESAMPLE, seeds, stratification | [HARD] |  |  |
| Q3 | Kruskal MST + union-find | [FAANG] |  |  |
| Q4 | Weighted reservoir sampling (A-Res) | [HARD] |  |  |
| Q5 | Spark 4 Python UDTF | [FAANG] |  |  |
| Q6 | Time-weighted averages (TWAP) | [HARD] |  |  |
| Q7 | Python worker memory model & Arrow | [HARD] |  |  |
| Q8 | State store: HDFS-backed vs RocksDB | [FAANG] |  |  |
| Q9 | DLQ replay with per-key FIFO ordering | [HARD] |  |  |
| Q10 | Poison-batch bisection | [FAANG] |  |  |
| Q11 | Delta row-level lineage (row tracking) | [FAANG] |  |  |
| Q12 | Non-events: expected-vs-actual coverage | [HARD] |  |  |
| Q13 | Heterogeneous multi-channel event union | [FAANG] |  |  |
| Q14 | SNOWFLAKE.ML FORECAST / ANOMALY_DETECTION | [FAANG] |  |  |
| Q15 | Marketplace listings & monetization math | [HARD] |  |  |
| Q16 | Metrics cardinality explosion | [HARD] |  |  |
| Q17 | MTTR analytics on job_run_timeline | [HARD] |  |  |
| Q18 | Usage-weighted data product ROI scorecard | [FAANG] |  |  |

**Day-34 lessons to remember**

* Build candidate sets from the **domain spine** (calendar slots), never from observed rows — observed-only spines hide the free/zero cases (Q1, Q12 — same lesson in SQL and modeling).
* Window-then-filter: compute `LEAD` over **all** rows before `WHERE`ing to failures (Q17) — a filtered window silently changes LEAD's input set.
* A-Res keeps the **largest** keys `u^(1/w)`; smallest-key selection inverts the weights (Q4).
* State-interval aggregates need **duration weighting**; the naive row-average is a different (wrong) number (Q6).
* MERGE sources must be **deduped per key**, and `WHEN MATCHED AND s.seq > t.seq` turns stale replays into no-ops (Q9).
* `TABLESAMPLE SYSTEM` raises on managed tables; BERNOULLI/PERCENT/ROWS work (Q2).
* Python UDTFs: `@udtf` + `eval` yields + `terminate` per partition; SQL `TABLE()` needs catalog registration — `DataFrame.lateralJoin()` always works (Q5).

**Next up — Day 35:** fresh topics only; candidates already reserved (not in Days 1–34): SQL sessionization with session gaps, adversarial fallbacks, tiny URL hints.